# 🎵 Mel-Band Roformer — Разделение аудио на стемы
### GitHub: https://github.com/chmyrega/
### Telegram: https://t.me/Y4TSK0V


---


Извлечение вокала, инструментала, караоке-версий и обработка аудио с помощью специализированных нейросетевых моделей.

**Перед началом:** убедитесь, что включён GPU → *Среда выполнения → Сменить среду → T4 GPU*

| Шаг | Действие |
|-----|----------|
| 1 | Выбрать модель |
| 2 | Установить зависимости и скачать модель |
| 3 | Загрузить аудиофайлы |
| 4 | Обработать |
| 5 | Прослушать и скачать результаты |

In [ ]:
#@title ▶️ Шаг 1: Выбор модели { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# ══════════════════════════════════════════════
#  Проверка GPU
# ══════════════════════════════════════════════
try:
    import torch
    if torch.cuda.is_available():
        _gpu = torch.cuda.get_device_name(0)
        _mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✅ GPU: {_gpu} ({_mem:.1f} GB)")
    else:
        print("⚠️  GPU НЕ ОБНАРУЖЕН! Среда выполнения → Сменить среду → GPU")
except ImportError:
    print("⚠️  PyTorch ещё не установлен (будет установлен на шаге 2)")

# ══════════════════════════════════════════════
#  Базовые URL
# ══════════════════════════════════════════════
GABOX = "https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main"
KIMB  = "https://huggingface.co/KimberleyJSN/melbandroformer/resolve/main"

CONFIG_URLS = {
    "default":       None,  # встроенный конфиг из репо
    "v10":           f"{GABOX}/melbandroformers/instrumental/v10.yaml",
    "v7":            f"{GABOX}/melbandroformers/vocals/v7.yaml",
    "karaoke":       f"{GABOX}/melbandroformers/karaoke/karaokegabox_1750911344.yaml",
    "karaoke_small": f"{GABOX}/melbandroformers/karaoke/config_karaoke_small.yaml",
}

# ══════════════════════════════════════════════
#  Каталог моделей: (имя, url, конфиг, размер_MB, описание)
# ══════════════════════════════════════════════
_C = {}  # {категория: [(name, url, cfg_key, size_mb, desc), ...]}

def _a(cat, name, url, cfg, sz, desc):
    _C.setdefault(cat, []).append((name, url, cfg, sz, desc))

# ── Инструментальные (стабильные) ──
_INS = f"{GABOX}/melbandroformers/instrumental"

_a("🎸 Инструментальные (стабильные)",
   "Inst GaboxF v9 ⭐ рекомендуется",
   f"{_INS}/Inst_GaboxFv9.ckpt", "default", 913,
   "Лучшее соотношение качества и стабильности. Чистые минусовки.")

_a("🎸 Инструментальные (стабильные)",
   "Inst GaboxF v8",
   f"{_INS}/Inst_GaboxFv8.ckpt", "default", 913,
   "Предыдущая проверенная версия.")

_a("🎸 Инструментальные (стабильные)",
   "Inst GaboxFVX",
   f"{_INS}/Inst_GaboxFVX.ckpt", "default", 913,
   "Версия VX — альтернатива v8/v9.")

_a("🎸 Инструментальные (стабильные)",
   "Inst Fv4",
   f"{_INS}/inst_Fv4.ckpt", "default", 913,
   "Стабильная, проверенная временем.")

_a("🎸 Инструментальные (стабильные)",
   "Inst GaboxF v7z",
   f"{_INS}/Inst_GaboxFv7z.ckpt", "default", 913,
   "Версия v7z.")

_a("🎸 Инструментальные (стабильные)",
   "INST V7N (шумоподавление)",
   f"{_INS}/INSTV7N.ckpt", "default", 913,
   "V7 со встроенным шумоподавлением. Для зашумлённых записей.")

_a("🎸 Инструментальные (стабильные)",
   "INST V6",
   f"{_INS}/INSTV6.ckpt", "default", 913,
   "Классическая модель V6.")

_a("🎸 Инструментальные (стабильные)",
   "INST V6N (шумоподавле��ие)",
   f"{_INS}/INSTV6N.ckpt", "default", 913,
   "V6 со встроенным шумоподавлением.")

_a("🎸 Инструментальные (стабильные)",
   "INST V5",
   f"{_INS}/INSTV5.ckpt", "default", 913,
   "Классическая модель V5.")

_a("🎸 Инструментальные (стабильные)",
   "Inst GaboxV7",
   f"{_INS}/Inst_GaboxV7.ckpt", "default", 913,
   "GaBox V7.")

# ── Инструментальные (компактные) ──
_a("🌸 Инструментальные (компактные / 490 MB)",
   "Inst Flowers V10 ⭐ быстрая и лёгкая",
   f"{_INS}/inst_gaboxFlowersV10.ckpt", "v10", 490,
   "Новая архитектура: вдвое легче, быстрее обработка. Использует конфиг v10.")

# ── Инструментальные (GaBox classic) ──
_a("🎸 Инструментальные (GaBox classic)",
   "inst_gabox (оригинал)",
   f"{_INS}/inst_gabox.ckpt", "default", 913,
   "Первая GaBox инструментальная модель.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gabox2",
   f"{_INS}/inst_gabox2.ckpt", "default", 913,
   "GaBox v2.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gabox3",
   f"{_INS}/inst_gabox3.ckpt", "default", 913,
   "GaBox v3.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxBv1",
   f"{_INS}/inst_gaboxBv1.ckpt", "default", 913,
   "GaBox B-серия v1.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxBv2",
   f"{_INS}/inst_gaboxBv2.ckpt", "default", 913,
   "GaBox B-серия v2.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxBv3",
   f"{_INS}/inst_gaboxBv3.ckpt", "default", 913,
   "GaBox B-серия v3.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxFv1",
   f"{_INS}/inst_gaboxFv1.ckpt", "default", 913,
   "GaBox F-серия v1.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxFv2",
   f"{_INS}/inst_gaboxFv2.ckpt", "default", 913,
   "GaBox F-серия v2.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxFv3",
   f"{_INS}/inst_gaboxFv3.ckpt", "default", 913,
   "GaBox F-серия v3.")

_a("🎸 Инструментальные (GaBox classic)",
   "intrumental_gabox (ранняя)",
   f"{_INS}/intrumental_gabox.ckpt", "default", 913,
   "Ранняя версия (оригинальная опечатка в имени файла).")

# ── Вокальные ──
_VOC = f"{GABOX}/melbandroformers/vocals"

_a("🎤 Вокальные",
   "Vocal Fv7 ⭐ новая архитектура",
   f"{_VOC}/voc_fv7.ckpt", "v7", 490,
   "Новейшая вокальная модель. Компактная (490 MB). Использует конфиг v7.")

_a("🎤 Вокальные",
   "Vocal Fv6",
   f"{_VOC}/voc_fv6.ckpt", "default", 913,
   "Стабильная вокальная модель v6.")

_a("🎤 Вокальные",
   "Vocal Fv5",
   f"{_VOC}/voc_fv5.ckpt", "default", 913,
   "Вокальная модель v5.")

_a("🎤 Вокальные",
   "Vocal Fv4 (voc_fv4)",
   f"{_VOC}/voc_fv4.ckpt", "default", 913,
   "Вокальная модель v4.")

_a("🎤 Вокальные",
   "Vocal Fv3",
   f"{_VOC}/voc_Fv3.ckpt", "default", 913,
   "Вокальная модель v3.")

_a("🎤 Вокальные",
   "Vocal GaBox Fv2",
   f"{_VOC}/voc_gaboxFv2.ckpt", "default", 913,
   "GaBox-серия вокальная Fv2.")

_a("🎤 Вокальные",
   "Vocal GaBox Fv1",
   f"{_VOC}/voc_gaboxFv1.ckpt", "default", 913,
   "GaBox-серия вокальная Fv1.")

_a("🎤 Вокальные",
   "Vocal GaBox 2",
   f"{_VOC}/voc_gabox2.ckpt", "default", 913,
   "GaBox вокальная v2.")

_a("🎤 Вокальные",
   "Vocal GaBox (оригинал)",
   f"{_VOC}/voc_gabox.ckpt", "default", 913,
   "Оригинальная GaBox вокальная модель.")

# ── Караоке ──
_KAR = f"{GABOX}/melbandroformers/karaoke"

_a("🎙️ Караоке",
   "Karaoke GaBox V1",
   f"{_KAR}/Karaoke_GaboxV1.ckpt", "karaoke", 913,
   "Удаляет ведущий вокал, сохраняет бэк-вокал и гармонии.")

_a("🎙️ Караоке",
   "Small Karaoke — быстрая (203 MB)",
   f"{_KAR}/small_karaoke_gaboxaufr.ckpt", "karaoke_small", 203,
   "Компактная модель. Быстро, но менее точно.")

# ── Экспериментальные ──
_EXP = f"{GABOX}/melbandroformers/experimental"

_a("🧪 Экспериментальные",
   "Inst Experimental V1",
   f"{_INS}/Inst_ExperimentalV1.ckpt", "default", 913,
   "Экспериментальная инструментальная модель.")

_a("🧪 Экспериментальные",
   "INST V10",
   f"{_EXP}/INSTV10.ckpt", "default", 913,
   "Инструментальная V10 (экспериментальная).")

_a("🧪 Экспериментальные",
   "INST V9",
   f"{_EXP}/INSTV9.ckpt", "default", 913,
   "Инструментальная V9 (экспериментальная).")

_a("🧪 Экспериментальные",
   "INST V8",
   f"{_EXP}/INSTV8.ckpt", "default", 913,
   "Инструментальная V8 (экспериментальная).")

_a("🧪 Экспериментальные",
   "INST V8N (шумоподавление)",
   f"{_EXP}/INSTV8N.ckpt", "default", 913,
   "V8 со встроенным шумоподавлением.")

_a("🧪 Экспериментальные",
   "Inst V7 Plus",
   f"{_EXP}/instv7plus.ckpt", "default", 913,
   "Улучшенная версия V7.")

_a("🧪 Экспериментальные",
   "Inst V7 Beta 1",
   f"{_EXP}/instv7beta.ckpt", "default", 913,
   "Бета 1 инструментальной V7.")

_a("🧪 Экспериментальные",
   "Inst V7 Beta 2",
   f"{_EXP}/instv7beta2.ckpt", "default", 913,
   "Бета 2 инструментальной V7.")

_a("🧪 Экспериментальные",
   "Inst V7 Beta 3",
   f"{_EXP}/instv7beta3.ckpt", "default", 913,
   "Бета 3 инструментальной V7.")

_a("🧪 Экспериментальные",
   "Inst Fv9 (exp)",
   f"{_EXP}/Inst_Fv9.ckpt", "default", 913,
   "Экспериментальная Fv9.")

_a("🧪 Экспериментальные",
   "Inst Fv8 (exp)",
   f"{_EXP}/Inst_Fv8.ckpt", "default", 913,
   "Экспериментальная Fv8.")

_a("🧪 Экспериментальные",
   "Inst Fv8b (exp)",
   f"{_EXP}/Inst_FV8b.ckpt", "default", 913,
   "Экспериментальная Fv8b.")

_a("🧪 Экспериментальные",
   "Inst fv7b (exp)",
   f"{_EXP}/inst_fv7b.ckpt", "default", 913,
   "Экспериментальная fv7b.")

_a("🧪 Экспериментальные",
   "Fullness",
   f"{_EXP}/Fullness.ckpt", "default", 913,
   "Модель «Полнота звука». Более насыщенный результат.")

_a("🧪 Экспериментальные",
   "Karaoke GaBox V2 (exp)",
   f"{_EXP}/Karaoke_GaboxV2.ckpt", "karaoke", 913,
   "Экспериментальная караоке V2.")

_a("🧪 Экспериментальные",
   "Lead Vocal Dereverb",
   f"{_EXP}/Lead_VocalDereverb.ckpt", "default", 913,
   "Удаление реверберации с ведущего вокала.")

_a("🧪 Экспериментальные",
   "Vocal Fv7 Beta 1",
   f"{_EXP}/vocfv7beta1.ckpt", "default", 913,
   "Бета 1 вокальной v7.")

_a("🧪 Экспериментальные",
   "Vocal Fv7 Beta 2",
   f"{_EXP}/vocfv7beta2.ckpt", "default", 913,
   "Бета 2 вокальной v7.")

_a("🧪 Экспериментальные",
   "Vocal Fv7 Beta 3",
   f"{_EXP}/vocfv7beta3.ckpt", "default", 913,
   "Бета 3 вокальной v7.")

_a("🧪 Экспериментальные",
   "kar_gabox (exp караоке)",
   f"{_EXP}/kar_gabox.ckpt", "default", 913,
   "Ранняя экспериментальная караоке.")

_a("🧪 Экспериментальные",
   "BS ResurrectioN",
   f"{_EXP}/BS_ResurrectioN.ckpt", "default", 204,
   "Маленькая экспериментальная модель (204 MB).")

_a("🧪 Экспериментальные",
   "small_inst (203 MB)",
   f"{_EXP}/small_inst.ckpt", "default", 203,
   "Маленькая быстрая инструментальная модель.")

# ── Специальные ──
_a("🔧 Специальные",
   "Denoise & Debleed",
   f"{_INS}/denoisedebleed.ckpt", "default", 913,
   "Шумоподавление + удаление перетекания между каналами.")

_a("🔧 Специальные",
   "INST V5N (шумоподавление)",
   f"{_INS}/INSTV5N.ckpt", "default", 913,
   "V5 со встроенным шумоподавлением.")

_a("🔧 Специальные",
   "Inst Fv4 Noise",
   f"{_INS}/inst_Fv4Noise.ckpt", "default", 913,
   "Fv4 с шумоподавлением.")

# ── Базовая (KimberleyJSN) ──
_a("📦 Базовая (KimberleyJSN)",
   "MelBandRoformer Original",
   f"{KIMB}/MelBandRoformer.ckpt", "default", 900,
   "Оригинальная модель из исходного репозитория.")

# ══════════════════════════════════════════════
#  Интерфейс
# ══════════════════════════════════════════════
categories = list(_C.keys())

cat_dd = widgets.Dropdown(
    options=categories,
    value=categories[0],
    description="Категория:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="600px"),
)

def _model_options(cat):
    return [(m[0], i) for i, m in enumerate(_C[cat])]

model_dd = widgets.Dropdown(
    options=_model_options(categories[0]),
    description="Модель:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="600px"),
)

info_out = widgets.Output()
confirm_out = widgets.Output()

mode_toggle = widgets.ToggleButtons(
    options=["📋 Из каталога", "🔗 Свой URL"],
    value="📋 Из каталога",
    style={"button_width": "150px"},
)

custom_url = widgets.Text(
    placeholder="https://huggingface.co/.../resolve/main/model.ckpt",
    description="URL модели:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="600px", display="none"),
)

custom_config = widgets.Dropdown(
    options=list(CONFIG_URLS.keys()),
    value="default",
    description="Конфиг:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="300px", display="none"),
)

def _show_info():
    with info_out:
        clear_output(wait=True)
        if mode_toggle.value == "🔗 Свой URL":
            print("📝 Введите прямую ссылку на .ckpt с HuggingFace")
            print("   и выберите конфиг (default для стандартных 913 MB моделей).")
            return
        cat = cat_dd.value
        idx = model_dd.value
        name, url, cfg, sz, desc = _C[cat][idx]
        print(f"📦 {name}")
        print(f"   Размер: ~{sz} MB  |  Конфиг: {cfg}")
        print(f"   📝 {desc}")
        print(f"   🔗 {url}")

def _on_cat(change):
    model_dd.options = _model_options(change["new"])
    model_dd.value = 0
    _show_info()

def _on_model(change):
    _show_info()

def _on_mode(change):
    is_custom = change["new"] == "🔗 Свой URL"
    cat_dd.layout.display = "none" if is_custom else "block"
    model_dd.layout.display = "none" if is_custom else "block"
    custom_url.layout.display = "block" if is_custom else "none"
    custom_config.layout.display = "block" if is_custom else "none"
    _show_info()

cat_dd.observe(_on_cat, names="value")
model_dd.observe(_on_model, names="value")
mode_toggle.observe(_on_mode, names="value")

confirm_btn = widgets.Button(
    description="✔ Подтвердить выбор",
    button_style="success",
    icon="check",
    layout=widgets.Layout(width="200px"),
)

def _on_confirm(b):
    with confirm_out:
        clear_output(wait=True)
        if mode_toggle.value == "🔗 Свой URL":
            url = custom_url.value.strip()
            if not url or not url.startswith("http"):
                print("❌ Введите корректный URL!")
                return
            cfg_key = custom_config.value
            sel_name = url.split("/")[-1]
        else:
            cat = cat_dd.value
            idx = model_dd.value
            sel_name, url, cfg_key, sz, desc = _C[cat][idx]

        global SELECTED_MODEL_URL, SELECTED_CONFIG_KEY, SELECTED_MODEL_NAME
        SELECTED_MODEL_URL = url
        SELECTED_CONFIG_KEY = cfg_key
        SELECTED_MODEL_NAME = sel_name

        print(f"✅ Выбрано: {sel_name}")
        print(f"   URL: {url}")
        print(f"   Конфиг: {cfg_key}")
        print(f"\n➡️  Переходите к Шагу 2!")

confirm_btn.on_click(_on_confirm)

print("🎯 Выберите модель для разделения аудио:\n")
display(mode_toggle)
display(cat_dd)
display(model_dd)
display(custom_url)
display(custom_config)
display(info_out)
display(confirm_btn)
display(confirm_out)
_show_info()

✅ GPU: Tesla T4 (15.6 GB)
🎯 Выберите модель для разделения аудио:



ToggleButtons(options=('📋 Из каталога', '🔗 Свой URL'), style=ToggleButtonsStyle(button_width='150px'), value='…

Dropdown(description='Категория:', layout=Layout(width='600px'), options=('🎸 Инструментальные (стабильные)', '…

Dropdown(description='Модель:', layout=Layout(width='600px'), options=(('Inst GaboxF v9 ⭐ рекомендуется', 0), …

Text(value='', description='URL модели:', layout=Layout(display='none', width='600px'), placeholder='https://h…

Dropdown(description='Конфиг:', layout=Layout(display='none', width='300px'), options=('default', 'v10', 'v7',…

Output()

Button(button_style='success', description='✔ Подтвердить выбор', icon='check', layout=Layout(width='200px'), …

Output()

## ⚙️ Шаг 2: Установка и загрузка модели
Запустите ячейку ниже. Установка занимает ~2-3 минуты (зависимости + скачивание модели).

In [ ]:
#@title ▶️ Шаг 2: Установка { display-mode: "form" }

import os, subprocess, shutil
from pathlib import Path
from IPython.display import clear_output

# ── Проверка выбора ──
try:
    _url = SELECTED_MODEL_URL
    _cfg = SELECTED_CONFIG_KEY
    _name = SELECTED_MODEL_NAME
    _cfg_urls = CONFIG_URLS  # из ячейки 1
except NameError:
    raise RuntimeError(
        "❌ Сначала запустите Шаг 1 и подтвердите выбор модели!\n"
        "   (Если ядро было перезапущено — нужно заново выполнить Шаг 1)"
    )

# ── Пути ──
REPO_DIR    = Path("/content/Mel-Band-Roformer-Vocal-Model")
MODEL_PATH  = REPO_DIR / "model.ckpt"
SONGS_DIR   = Path("/content/songs")
RESULTS_DIR = Path("/content/results")
DEFAULT_CFG = REPO_DIR / "configs" / "config_vocals_mel_band_roformer.yaml"

# ── 1. Клонирование ──
print("📂 Репозиторий...")
if REPO_DIR.exists():
    print("   Уже существует, пропускаем.")
else:
    ret = subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/KimberleyJensen/Mel-Band-Roformer-Vocal-Model",
         str(REPO_DIR)],
        capture_output=True, text=True
    )
    if ret.returncode != 0:
        raise RuntimeError(f"❌ git clone:\n{ret.stderr}")
    print("   ✓ Склонирован.")

# ── 2. Зависимости (показываем ошибки) ──
print("📦 Зависимости...")
ret = subprocess.run(
    ["pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print(f"⚠️  Предупреждение pip:\n{ret.stderr[:500]}")
else:
    print("   ✓ requirements.txt")

ret = subprocess.run(
    ["pip", "install", "-q", "librosa", "pydub"],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print(f"⚠️  pip librosa/pydub:\n{ret.stderr[:300]}")
else:
    print("   ✓ librosa, pydub")

# ── 3. Свободное место ──
stat = shutil.disk_usage("/content")
free_gb = stat.free / 1e9
print(f"💾 Свободно: {free_gb:.1f} GB")
if free_gb < 2.0:
    print("   ⚠️ Мало места! Рекомендуется ≥2 GB.")

# ── 4. Скачивание модели ──
print(f"\n🤖 Модель: {_name}")
print(f"   {_url}")

if MODEL_PATH.exists():
    MODEL_PATH.unlink()

ret = subprocess.run(
    ["wget", "--progress=dot:giga", "-O", str(MODEL_PATH), _url],
)
if ret.returncode != 0 or not MODEL_PATH.exists():
    raise RuntimeError("❌ Не удалось скачать модель. Проверьте URL.")

file_size_mb = MODEL_PATH.stat().st_size / 1e6
print(f"   Размер: {file_size_mb:.0f} MB")

# Защита от HTML вместо модели
if file_size_mb < 10:
    with open(MODEL_PATH, "rb") as f:
        head = f.read(200)
    if b"<html" in head.lower() or b"<!doctype" in head.lower():
        MODEL_PATH.unlink()
        raise RuntimeError("❌ Вместо модели скачалась HTML-страница. URL недействителен.")
    print("   ⚠️ Файл подозрительно мал.")

# Проверка через PyTorch
import torch
try:
    _test = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
    del _test  # ← освобождаем память!
    import gc; gc.collect()
    print("   ✓ Проверка PyTorch пройдена.")
except Exception as e:
    raise RuntimeError(f"❌ Модель повреждена: {e}")

# ── 5. Конфиг ──
CONFIG_PATH = DEFAULT_CFG
if _cfg != "default":
    cfg_url = _cfg_urls.get(_cfg)
    if cfg_url:
        cfg_dest = REPO_DIR / f"config_{_cfg}.yaml"
        print(f"\n📄 Конфиг: {_cfg}")
        ret = subprocess.run(
            ["wget", "-q", "-O", str(cfg_dest), cfg_url],
            capture_output=True, text=True
        )
        if ret.returncode == 0 and cfg_dest.exists() and cfg_dest.stat().st_size > 50:
            CONFIG_PATH = cfg_dest
            print(f"   ✓ {cfg_dest.name}")
        else:
            print(f"   ⚠️ Не удалось скачать конфиг, используем default.")
    else:
        print(f"   ⚠️ URL конфига '{_cfg}' не найден, используем default.")

# ── 6. Рабочие папки ──
SONGS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# ── 7. Глобальные переменные для следующих шагов ──
ACTIVE_CONFIG_PATH = str(CONFIG_PATH)
ACTIVE_MODEL_PATH  = str(MODEL_PATH)

# ── Итог (clear с wait=True чтобы не мигало) ──
clear_output(wait=True)
print("=" * 55)
print(f"✅ Установка завершена!")
print(f"   Модель:  {_name}")
print(f"   Конфиг:  {CONFIG_PATH.name}")
print(f"   Размер:  {file_size_mb:.0f} MB")
print(f"   Диск:    {free_gb:.1f} GB свободно")
print("=" * 55)
print("\n📋 Форматы: MP3, WAV, FLAC, M4A, OGG, WMA, AIFF, OPUS")
print("➡️  Переходите к Шагу 3!")

✅ Установка завершена!
   Модель:  Inst GaboxF v9 ⭐ рекомендуется
   Конфиг:  config_vocals_mel_band_roformer.yaml
   Размер:  913 MB
   Диск:    74.3 GB свободно

📋 Форматы: MP3, WAV, FLAC, M4A, OGG, WMA, AIFF, OPUS
➡️  Переходите к Шагу 3!


## 📤 Шаг 3: Загрузка аудиофайлов
Выберите способ загрузки: с компьютера или из Google Drive.

In [ ]:
#@title ▶️ Шаг 3: Загрузка файлов { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import shutil

SONGS_DIR = Path("/content/songs")
SONGS_DIR.mkdir(exist_ok=True)  # ← на случай если шаг 2 не запускался

# Очистка от предыдущих загрузок (с проверкой существования)
if SONGS_DIR.exists():
    for f in SONGS_DIR.iterdir():
        f.unlink()

AUDIO_EXT = {".mp3", ".wav", ".flac", ".m4a", ".ogg", ".wma",
             ".aiff", ".aif", ".opus", ".webm"}

mode = widgets.ToggleButtons(
    options=["💻 С компьютера", "📁 Google Drive"],
    value="💻 С компьютера",
    style={"button_width": "160px"},
)

gdrive_path = widgets.Text(
    value="/content/drive/MyDrive/music",
    description="Путь в Drive:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="600px", display="none"),
)

upload_out = widgets.Output()

def _on_mode_change(change):
    gdrive_path.layout.display = (
        "block" if change["new"] == "📁 Google Drive" else "none"
    )

mode.observe(_on_mode_change, names="value")

go_btn = widgets.Button(
    description="📥 Загрузить", button_style="primary", icon="upload"
)

def _on_go(b):
    with upload_out:
        clear_output(wait=True)

        if mode.value == "💻 С компьютера":
            from google.colab import files as colab_files
            print("📤 Выберите аудиофайлы (можно несколько):\n")
            uploaded = colab_files.upload()
            for fname in uploaded:
                src = Path(fname)
                if src.suffix.lower() not in AUDIO_EXT:
                    print(f"   ⚠️ Пропущен (неизвестный формат): {fname}")
                    src.unlink(missing_ok=True)
                    continue
                dest = SONGS_DIR / fname
                shutil.move(str(src), str(dest))
                print(f"   ✓ {fname} ({dest.stat().st_size / 1e6:.1f} MB)")

        else:
            from google.colab import drive
            if not Path("/content/drive").exists():
                print("🔗 Подключение Google Drive...")
                drive.mount("/content/drive")

            src = Path(gdrive_path.value)
            if not src.exists():
                print(f"❌ Путь не найден: {src}")
                return

            files_found = []
            if src.is_file():
                files_found = [src]
            else:
                for ext in AUDIO_EXT:
                    files_found.extend(src.glob(f"*{ext}"))
                    files_found.extend(src.glob(f"*{ext.upper()}"))

            files_found = sorted(set(files_found))
            if not files_found:
                print(f"❌ Аудиофайлы не найдены в {src}")
                return

            for fp in files_found:
                dest = SONGS_DIR / fp.name
                shutil.copy2(str(fp), str(dest))
                print(f"   ✓ {fp.name} ({fp.stat().st_size / 1e6:.1f} MB)")

        total = list(SONGS_DIR.iterdir())
        if total:
            print(f"\n📁 Всего файлов: {len(total)}")
            print("➡️  Переходите к Шагу 4!")
        else:
            print("\n❌ Ни одного файла не загружено.")

go_btn.on_click(_on_go)

display(mode)
display(gdrive_path)
display(go_btn)
display(upload_out)

ToggleButtons(options=('💻 С компьютера', '📁 Google Drive'), style=ToggleButtonsStyle(button_width='160px'), va…

Text(value='/content/drive/MyDrive/music', description='Путь в Drive:', layout=Layout(display='none', width='6…

Button(button_style='primary', description='📥 Загрузить', icon='upload', style=ButtonStyle())

Output()

## 🚀 Шаг 4: Обработка
Конвертация в WAV (при необходимости) → разделение на стемы.

In [ ]:
#@title ▶️ Шаг 4: Обработка { display-mode: "form" }

import subprocess, time, os, glob
from pathlib import Path

SONGS_DIR   = Path("/content/songs")
RESULTS_DIR = Path("/content/results")
REPO_DIR    = Path("/content/Mel-Band-Roformer-Vocal-Model")

# ── Проверки ──
audio_files = list(SONGS_DIR.iterdir())
if not audio_files:
    raise RuntimeError("❌ Нет файлов для обработки. Вернитесь к Шагу 3.")

try:
    _ = ACTIVE_MODEL_PATH, ACTIVE_CONFIG_PATH
except NameError:
    raise RuntimeError("❌ Модель не установлена. Вернитесь к Шагу 2.")

# ── Конвертация не-WAV файлов в WAV через ffmpeg ──
print("🔄 Подготовка файлов...")
wav_count = 0
for f in list(SONGS_DIR.iterdir()):
    if f.suffix.lower() == ".wav":
        wav_count += 1
        continue
    wav_path = f.with_suffix(".wav")
    print(f"   Конвертация: {f.name} → {wav_path.name}")
    ret = subprocess.run(
        ["ffmpeg", "-y", "-i", str(f), "-ar", "44100", "-ac", "2",
         str(wav_path)],
        capture_output=True, text=True
    )
    if ret.returncode == 0 and wav_path.exists():
        f.unlink()  # удаляем оригинал, оставляем WAV
        wav_count += 1
    else:
        print(f"   ⚠️ Не удалось конвертировать {f.name}: {ret.stderr[:200]}")

wav_files = list(SONGS_DIR.glob("*.wav"))
if not wav_files:
    raise RuntimeError("❌ Нет WAV-файлов для обработки.")

print(f"   ✓ Готово WAV-файлов: {len(wav_files)}\n")

# ── Очистка результатов ──
if RESULTS_DIR.exists():
    for f in RESULTS_DIR.iterdir():
        f.unlink()
RESULTS_DIR.mkdir(exist_ok=True)

# ── Запуск обработки ──
print(f"🎵 Модель: {SELECTED_MODEL_NAME}")
print(f"📄 Конфиг: {Path(ACTIVE_CONFIG_PATH).name}")
print("━" * 55)

start = time.time()

ret = subprocess.run(
    ["python", "inference.py",
     "--config_path", ACTIVE_CONFIG_PATH,
     "--model_path",  ACTIVE_MODEL_PATH,
     "--input_folder", str(SONGS_DIR),
     "--store_dir",    str(RESULTS_DIR)],
    cwd=str(REPO_DIR),
    text=True
)

elapsed = time.time() - start

print("━" * 55)

results = list(RESULTS_DIR.iterdir())
if ret.returncode != 0 or not results:
    print(f"❌ Обработка завершилась с ошибкой (код {ret.returncode}).")
    print("   Возможные причины:")
    print("   • Неподходящий конфиг для выбранной модели")
    print("   • Недостаточно памяти GPU")
    print("   • Повреждённый аудиофайл")
else:
    print(f"✅ Обработка завершена за {elapsed:.0f} сек ({elapsed/60:.1f} мин)")
    print(f"\n📂 Результаты ({len(results)} файлов):")
    for r in sorted(results):
        sz = r.stat().st_size / 1e6
        print(f"   • {r.name}  ({sz:.1f} MB)")
    print("\n➡️  Переходите к Шагу 5!")

🔄 Подготовка файлов...
   Конвертация: plenka-observer-remakeslowed.mp3 → plenka-observer-remakeslowed.wav
   ✓ Готово WAV-файлов: 1

🎵 Модель: Inst GaboxF v9 ⭐ рекомендуется
📄 Конфиг: config_vocals_mel_band_roformer.yaml
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ Обработка завершена за 35 сек (0.6 мин)

📂 Результаты (2 файлов):
   • plenka-observer-remakeslowed_instrumental.wav  (65.1 MB)
   • plenka-observer-remakeslowed_vocals.wav  (65.1 MB)

➡️  Переходите к Шагу 5!


## 🎧 Шаг 5: Прослушивание и скачивание
Послушайте результаты прямо в браузере, затем скачайте ZIP-архив.

In [ ]:
#@title ▶️ Шаг 5: Прослушать и скачать { display-mode: "form" }

import IPython.display as ipd
from pathlib import Path
from datetime import datetime
import zipfile, re

RESULTS_DIR = Path("/content/results")
results = sorted(RESULTS_DIR.glob("*.wav"))

if not results:
    print("❌ Нет результатов. Сначала выполните Шаг 4.")
else:
    # ── Превью ──
    print("🎧 Прослушивание результатов:\n")
    for r in results:
        if "vocal" in r.name.lower():
            emoji = "🎤"
        elif "instrument" in r.name.lower():
            emoji = "🎸"
        else:
            emoji = "🎵"
        print(f"{emoji} {r.name}")
        # НЕ указываем rate= — IPython прочитает sr из заголовка WAV
        display(ipd.Audio(str(r)))
        print()

    # ── ZIP ──
    print("━" * 55)
    try:
        _sel_name = SELECTED_MODEL_NAME
    except NameError:
        _sel_name = "results"

    safe_name = re.sub(r'[^\w\-]', '_', _sel_name)
    safe_name = re.sub(r'_+', '_', safe_name).strip('_') or "results"
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    zip_name = f"{safe_name}_{ts}.zip"
    zip_path = Path(f"/content/{zip_name}")

    print(f"📦 Архив: {zip_name}")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for r in results:
            zf.write(r, r.name)
    print(f"   Размер: {zip_path.stat().st_size / 1e6:.1f} MB")

    from google.colab import files as colab_files
    print("\n⬇️  Скачивание...")
    colab_files.download(str(zip_path))

    print("\n💡 Для сохранения на Google Drive:")
    print(f'   !cp "{zip_path}" "/content/drive/MyDrive/"')

In [ ]:
#@title 🔄 Быстрая смена модели (без переустановки) { display-mode: "form" }

import subprocess
from pathlib import Path

REPO_DIR   = Path("/content/Mel-Band-Roformer-Vocal-Model")
MODEL_PATH = REPO_DIR / "model.ckpt"
DEFAULT_CFG = REPO_DIR / "configs" / "config_vocals_mel_band_roformer.yaml"

try:
    _url = SELECTED_MODEL_URL
    _cfg = SELECTED_CONFIG_KEY
    _name = SELECTED_MODEL_NAME
    _cfg_urls = CONFIG_URLS
except NameError:
    raise RuntimeError("❌ Сначала выберите модель в Шаге 1!")

if not REPO_DIR.exists():
    raise RuntimeError("❌ Репозиторий не найден. Выполните Шаг 2 полностью.")

print(f"🔄 Загрузка: {_name}")
if MODEL_PATH.exists():
    MODEL_PATH.unlink()

ret = subprocess.run(
    ["wget", "--progress=dot:giga", "-O", str(MODEL_PATH), _url],
)
if ret.returncode != 0 or not MODEL_PATH.exists():
    raise RuntimeError("❌ Не удалось скачать модель.")

# Проверка
import torch, gc
try:
    _t = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
    del _t; gc.collect()
except Exception as e:
    raise RuntimeError(f"❌ Модель повреждена: {e}")

# Конфиг
CONFIG_PATH = DEFAULT_CFG
if _cfg != "default":
    cfg_url = _cfg_urls.get(_cfg)
    if cfg_url:
        cfg_dest = REPO_DIR / f"config_{_cfg}.yaml"
        subprocess.run(["wget", "-q", "-O", str(cfg_dest), cfg_url], capture_output=True)
        if cfg_dest.exists() and cfg_dest.stat().st_size > 50:
            CONFIG_PATH = cfg_dest

ACTIVE_CONFIG_PATH = str(CONFIG_PATH)
ACTIVE_MODEL_PATH  = str(MODEL_PATH)

file_size_mb = MODEL_PATH.stat().st_size / 1e6
print(f"\n✅ {_name} ({file_size_mb:.0f} MB) | Конфиг: {CONFIG_PATH.name}")
print("➡️  Переходите к Шагу 4!")

🔄 Загрузка: Inst GaboxF v9 ⭐ рекомендуется

✅ Inst GaboxF v9 ⭐ рекомендуется (913 MB) | Конфиг: config_vocals_mel_band_roformer.yaml
➡️  Переходите к Шагу 4!
